# Interview Starter — Data Science and Machine Learning

Work from **main/** with the **main/.venv** kernel. This notebook composes the copied ThreeWToolkit configurations and leaves space for your own decisions and code.

**Start:** write the problem statement → run imports → fill Configuration → select a data source → inspect quality → choose a split → enable a modeling branch.

With no data selected, **Run All completes without loading interview data or training**. Imports may initialize local toolkit logs. The optional synthetic example tests wiring only; its scores are not evidence of performance on real data.

## Contents
1. Problem and assumptions
2. Environment and import library
3. Configuration and data loading: CSV, Parquet, JSON, Excel, SQLite, SQL
4. Data quality, schema, and exploration
5. Toolkit event adapter and leakage-aware splitting
6. Toolkit preprocessing, time-window features, Random Forest, and MLP
7. Evaluation, clustering, anomaly detection, and time-series diagnostics
8. Alternative tabular classification/regression
9. Persistence, reporting, and interview notes

[Workspace guide](AGENTS.md) · [Toolkit API guide](ThreeWToolkit/AGENTS.md) · [Environment setup](README.md)

## 1. Problem and assumptions

Fill in before modeling:
- Business question and decision supported:
- Prediction target, prediction time, and horizon:
- Observation grain: row, event, well, customer, or another entity:
- Feature meanings, units, and availability at prediction time:
- Dataset origin, permission to use, and size:
- Train/validation/test strategy and primary metric:
- Baseline and expected failure costs:

**Your notes / code:**

## 2. Environment and import library

These imports follow the current toolkit examples. They intentionally avoid older v2 import paths and wildcard imports.
If a dependency is missing, install the project into this notebook's environment using the README. Optional SQLAlchemy and Excel-engine imports occur only when those sources are selected.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.metadata
import inspect
import json
import sqlite3

WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / "ThreeWToolkit").is_dir() and (WORKSPACE / "main/ThreeWToolkit").is_dir():
    WORKSPACE = WORKSPACE / "main"
if not (WORKSPACE / "ThreeWToolkit").is_dir():
    raise RuntimeError("Launch this notebook from main/ or the repository root.")

cache_dir = WORKSPACE / ".cache/matplotlib"
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from IPython.display import display
import joblib

import ThreeWToolkit
package_file = Path(ThreeWToolkit.__file__).resolve()
assert package_file.is_relative_to(WORKSPACE / "ThreeWToolkit"), (
    "Wrong toolkit copy imported. Select main/.venv and install main in editable mode."
)
print("Python:", sys.executable)
print("Toolkit:", package_file)
print("Version:", importlib.metadata.version("ThreeWToolkit"))

In [ ]:
# Dataset contracts, loaders, transformations, and preprocessing.
from ThreeWToolkit.core import BaseDataset, DatasetOutputs
from ThreeWToolkit.core.base_trainer import TrainingResult, PredictionResult
from ThreeWToolkit.core.enums import TaskTypeEnum, DistanceMetricEnum, LinkageMethodEnum
from ThreeWToolkit.dataset import (
    ParquetDataset, ParquetDatasetConfig, load_csv, SubsetDataset,
    TransformConfig, TransformDataset, TransformedDataset,
)
from ThreeWToolkit.preprocessing import (
    CleanSignals, CleanSignalsConfig, ImputeMissing, ImputeMissingConfig,
    Normalize, NormalizeConfig, RenameColumns, RenameColumnsConfig,
    FillLabels, FillLabelsConfig, RemapClass, RemapClassConfig,
    SequentialPreprocessingAdapter, SequentialPreprocessingAdapterConfig,
)
from ThreeWToolkit.feature_extraction import (
    Windowing, WindowingConfig, StatisticalFeatures, StatisticalConfig,
    EWStatisticalFeatures, EWStatisticalConfig, WaveletFeatures, WaveletConfig,
    SequentialFeatureAdapter, SequentialFeatureAdapterConfig,
    ConcatFeatureAdapter, ConcatFeatureAdapterConfig,
)

In [ ]:
from ThreeWToolkit.models import MLP, MLPConfig, SklearnModels, SklearnModelsConfig
from ThreeWToolkit.models.torch_models import TorchModels, TorchModelsConfig
from ThreeWToolkit.trainer import (
    SklearnTrainer, SklearnTrainerConfig, TorchTrainer, TorchTrainerConfig,
)
from ThreeWToolkit.assessment import (
    ModelAssessment, ModelAssessmentConfig,
    AssessmentVisualization, AssessmentVisualizationConfig,
)
from ThreeWToolkit import metrics as toolkit_metrics
from ThreeWToolkit.pipeline import Pipeline as ToolkitPipeline, PipelineConfig
from ThreeWToolkit.utils import (
    TrainTestSplitter, KFoldSplitter, ModelRecorder,
    get_figshare_data, get_config_dataset_ini, load_config_in_dataset_ini,
)
from ThreeWToolkit.reports import ReportGeneration

In [ ]:
from ThreeWToolkit.data_visualization import (
    DataVisualization, PlotSeries, PlotMultipleSeries, CorrelationHeatmap,
    PlotFFT, SeasonalDecompositionPlot, WaveletSpectrogramPlot, ThreeWChart,
    DataQualityHeatmap, DendrogramPlot, ClusterSizeCurvePlot, SelectionHeatmapPlot,
    ClusteringOverlayPlot, RankedDistancePlot,
)
from ThreeWToolkit.clustering import (
    InstanceQualityFilter, TimeSeriesResampler, TimeSeriesScaler, DistanceComputer,
    HierarchicalClusterer, DivisiveRanker, MultivariateConsensus, compute_dba_centroid,
)
# Clustering configs are in this module, not ThreeWToolkit.clustering.
from ThreeWToolkit.core.base_clustering import (
    InstanceQualityConfig, ResamplingConfig, TimeSeriesScalingConfig,
    DistanceMatrixConfig, HierarchicalClusteringConfig, DivisiveClusteringConfig,
    MultivariateConsensusConfig,
)

In [ ]:
# Complementary sklearn tools; these are not part of ThreeWToolkit.
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, IsolationForest
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import (
    train_test_split, GroupShuffleSplit, TimeSeriesSplit, StratifiedKFold,
)
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score, silhouette_score,
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN

def config_reference(config_class):
    """Inspect current Pydantic parameters instead of guessing from old tutorials."""
    return pd.DataFrame([
        {"parameter": name, "required": field.is_required(),
         "default": str(field.default), "description": field.description}
        for name, field in config_class.model_fields.items()
    ])

# Uncomment to inspect a configuration or function.
# display(config_reference(WindowingConfig))
# print(inspect.signature(DataVisualization.plot_fft))

## 3. Configuration

Fill in these placeholders for interview data. Feature columns must exclude the target, IDs, and any values unavailable at prediction time.

**Sources:** csv, parquet, json, jsonl, excel, sqlite, sql, synthetic. For a folder of Parquet files, first select one file or explicitly combine compatible files while retaining event IDs.

**Workflows:** events uses ThreeWToolkit on numeric sequences; tabular uses an ordinary sklearn pipeline for mixed column types or regression.

**Splits:** group holds out entire entities; time holds out later records/events; random is for genuinely independent observations. Do not choose random just to get a higher score.

To try the clearly labeled toy example, set DATA_SOURCE to "synthetic" and RUN_TRAINING to True. It supplies example columns and an event-group split only when you have not supplied them.

In [ ]:
SEED = 42
DATA_SOURCE = None
DATA_PATH = None                 # Path relative to main/, or an absolute local path
SQL_QUERY = None                 # A SELECT query, preferably with bound parameters
SQL_PARAMS = {}
SQL_URL_ENV = "INTERVIEW_DATABASE_URL"
EXCEL_SHEET = 0
CSV_SEPARATOR = ","

WORKFLOW = "events"              # "events" or "tabular"
TASK = "classification"          # regression is implemented in the tabular branch
TARGET_COLUMN = None
TIMESTAMP_COLUMN = None
EVENT_COLUMN = None              # one uninterrupted sequence
GROUP_COLUMN = None              # well/customer/entity to keep out of other splits
FEATURE_COLUMNS = []
SPLIT_STRATEGY = None            # "group", "time", or "random"; choose deliberately
TEST_FRACTION = 0.2
VALIDATION_FRACTION = 0.2        # fraction of the remaining development set

RUN_TRAINING = False
RUN_TORCH = False
RUN_CLUSTERING = False
RUN_ANOMALY_DETECTION = False
RUN_SIGNAL_DIAGNOSTICS = False
SAVE_ARTIFACTS = False
GENERATE_HTML_REPORT = False

WINDOW_SIZE = 32
WINDOW_OVERLAP = 0.0
USE_SIGNAL_CLEANING = False      # enable after examining signal quality
USE_NORMALIZATION = False       # norm scaling can remove useful absolute levels
USE_WAVELETS = False
USE_EXPONENTIAL_FEATURES = False
MLP_EPOCHS = 10
SAMPLE_RATE_HZ = None            # needed for physically meaningful FFT frequencies
SEASONAL_PERIOD = None           # in samples; requires a defensible period
CLUSTER_SENSOR = None
CLUSTER_EVENT_LIMIT = 50         # DTW pairwise distances can be expensive

OUTPUT_DIR = WORKSPACE / "output/interview"
np.random.seed(SEED)
torch.manual_seed(SEED)

### File and SQL loaders

CSV/Parquet/JSON use pandas. Excel needs an engine such as openpyxl (available through the project's images extra). SQLite uses Python's built-in driver and opens an existing database read-only. Other SQL databases need SQLAlchemy and the matching driver, plus a connection URL in the named environment variable. Use an account with read-only access; do not put credentials in notebook cells.

The toolkit's **load_csv** helper is also imported; its arguments are file_path, column_names, date_column, and parse_dates. It returns a DataFrame, not a toolkit event dataset.

In [ ]:
def resolve_local_path(path):
    if path is None:
        raise ValueError("Set DATA_PATH.")
    result = Path(path).expanduser()
    result = result if result.is_absolute() else WORKSPACE / result
    if not result.is_file():
        raise FileNotFoundError(result)
    return result

def read_interview_table(source, path=None, query=None, params=None):
    if source == "sql":
        if not query:
            raise ValueError("Set SQL_QUERY to a read-only query.")
        try:
            from sqlalchemy import create_engine, text
        except ImportError as exc:
            raise ImportError("Install SQLAlchemy and your database driver in main/.venv.") from exc
        url = os.environ.get(SQL_URL_ENV)
        if not url:
            raise ValueError(f"Set environment variable {SQL_URL_ENV}; do not print its value.")
        engine = create_engine(url)
        try:
            with engine.connect() as connection:
                return pd.read_sql_query(text(query), connection, params=params or {})
        finally:
            engine.dispose()

    local_path = resolve_local_path(path)
    if source == "csv":
        return pd.read_csv(local_path, sep=CSV_SEPARATOR)
    if source == "parquet":
        return pd.read_parquet(local_path)
    if source in {"json", "jsonl"}:
        return pd.read_json(local_path, lines=(source == "jsonl"))
    if source == "excel":
        return pd.read_excel(local_path, sheet_name=EXCEL_SHEET)
    if source == "sqlite":
        if not query:
            raise ValueError("Set SQL_QUERY.")
        with sqlite3.connect(local_path.resolve().as_uri() + "?mode=ro", uri=True) as connection:
            return pd.read_sql_query(query, connection, params=params or {})
    raise ValueError(f"Unsupported DATA_SOURCE: {source}")

def make_synthetic_events(seed=42):
    """Toy signals for testing code only; these are not oil-well measurements."""
    rng = np.random.default_rng(seed)
    frames = []
    for event_number in range(20):
        label = event_number % 2
        t = np.arange(96)
        frame = pd.DataFrame({
            "event_id": f"demo_{event_number:02d}",
            "timestamp": pd.date_range("2025-01-01", periods=96, freq="s")
                         + pd.Timedelta(days=event_number),
            "sensor_a": 10 + 3 * label + np.sin(t / 8) + rng.normal(0, 0.2, len(t)),
            "sensor_b": 25 + label + np.cos(t / 11) + rng.normal(0, 0.3, len(t)),
            "target": label,
        })
        frame.loc[[10, 30], "sensor_a"] = np.nan
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

In [ ]:
raw_df = None
if DATA_SOURCE == "synthetic":
    raw_df = make_synthetic_events(SEED)
    TARGET_COLUMN = TARGET_COLUMN or "target"
    TIMESTAMP_COLUMN = TIMESTAMP_COLUMN or "timestamp"
    EVENT_COLUMN = EVENT_COLUMN or "event_id"
    GROUP_COLUMN = GROUP_COLUMN or "event_id"
    FEATURE_COLUMNS = FEATURE_COLUMNS or ["sensor_a", "sensor_b"]
    SPLIT_STRATEGY = SPLIT_STRATEGY or "group"
    print("SYNTHETIC DEMONSTRATION — not interview data or a scientific benchmark.")
elif DATA_SOURCE is not None:
    raw_df = read_interview_table(DATA_SOURCE, DATA_PATH, SQL_QUERY, SQL_PARAMS)

if raw_df is None:
    print("No data selected. Fill Configuration, then restart and run the notebook.")
else:
    print("Rows and columns:", raw_df.shape)
    display(raw_df.head())

### Existing 3W data: alternative loader reference

For a complete local 3W dataset, the native configuration is:

    native_config = ParquetDatasetConfig(
        path="/absolute/path/to/complete/3W/dataset",
        event_type=["real"], target_class=[0, 3, 4, 8],
        force_download=False,
    )
    # native_dataset = native_config.build()

This is intentionally not executed: the native loader validates the full 3W structure and may download when incomplete, even with force_download=False. Use pandas for a small sample or unfamiliar Parquet file. Download helpers are imported for reference but never called by this notebook.

**Your custom source / join / query:**

## 4. Data quality and schema

Inspect column types, missingness, duplicates, invalid dates, sampling intervals, and label coverage before changing anything. The copy below preserves raw_df. Duplicate observations are reported, not silently deleted. Missing target labels must be investigated before supervised learning; do not fill them just to make a model run.

In [ ]:
df = None
if raw_df is not None:
    df = raw_df.copy()
    if not df.columns.is_unique:
        raise ValueError("Duplicate column names: resolve before selecting features.")
    required = FEATURE_COLUMNS + [
        value for value in [TARGET_COLUMN, TIMESTAMP_COLUMN, EVENT_COLUMN, GROUP_COLUMN]
        if value is not None
    ]
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"Configured columns absent: {missing}")
    if TIMESTAMP_COLUMN:
        df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors="raise", utc=True)
        if df[TIMESTAMP_COLUMN].isna().any():
            raise ValueError("Missing timestamps: decide how to handle them.")
        sort_columns = ([EVENT_COLUMN] if EVENT_COLUMN else []) + [TIMESTAMP_COLUMN]
        df = df.sort_values(sort_columns, kind="stable").reset_index(drop=True)
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_fraction": df.isna().mean(),
        "unique_values": df.nunique(dropna=True),
    }))
    print("Duplicate rows:", int(df.duplicated().sum()))
    display(df.describe(include="all").T)
    if TARGET_COLUMN:
        display(df[TARGET_COLUMN].value_counts(dropna=False).rename("target_count"))

### Exploration and visualization

Plot one event at a time to avoid connecting unrelated sequences. Correlation is descriptive and does not establish causation. Use training data for subsequent feature-selection decisions; avoid repeatedly using final test outcomes to tune choices.

In [ ]:
example_df = None
if df is not None and FEATURE_COLUMNS:
    example_df = df
    if EVENT_COLUMN:
        example_id = df[EVENT_COLUMN].dropna().iloc[0]
        example_df = df.loc[df[EVENT_COLUMN] == example_id].copy()
    if TIMESTAMP_COLUMN:
        example_df = example_df.set_index(TIMESTAMP_COLUMN)
    numeric_features = [
        column for column in FEATURE_COLUMNS
        if pd.api.types.is_numeric_dtype(example_df[column])
    ]
    for column in numeric_features[:3]:
        fig, ax = DataVisualization.plot_series(
            example_df[column], title=f"Example event: {column}",
            xlabel=TIMESTAMP_COLUMN or "Row", ylabel=column,
        )
        plt.show()
        plt.close(fig)
    if len(numeric_features) >= 2:
        fig, ax = DataVisualization.correlation_heatmap(
            example_df[numeric_features], annot=True
        )
        plt.show()
        plt.close(fig)

**Your observations, domain interpretation, and additional plots:**

## 5. Adapt tables to toolkit events

The toolkit expects a BaseDataset whose items are DatasetOutputs(signal, label, metadata). The adapter below implements this small interface for arbitrary interview tables.

Each event must be an uninterrupted numeric sequence; EVENT_COLUMN separates events, while GROUP_COLUMN can identify a well containing several events. Labels and timestamps align with signal rows. IDs and the target are excluded from features. Classification labels may be strings or integers; RemapClassConfig handles encoding later.

In [ ]:
class FrameDataset(BaseDataset):
    def __init__(self, events):
        self.events = list(events)
        if not self.events:
            raise ValueError("No events available.")

    def __len__(self):
        return len(self.events)

    def __getitem__(self, idx):
        event = self.events[idx]
        # Protect original observations from transforms that modify their inputs.
        return DatasetOutputs(
            signal=event.signal.copy(),
            label=None if event.label is None else event.label.copy(),
            metadata=event.metadata.copy(),
        )

def events_from_frame(frame, features, target, event_column, time_column, group_column):
    if not features:
        raise ValueError("Set FEATURE_COLUMNS explicitly.")
    excluded = {v for v in [target, event_column, time_column, group_column] if v}
    if excluded.intersection(features):
        raise ValueError("Features contain target, ID, or timestamp columns.")
    if not event_column or not time_column:
        raise ValueError("Event workflow requires EVENT_COLUMN and TIMESTAMP_COLUMN.")
    if frame[event_column].isna().any():
        raise ValueError("Event IDs cannot be missing.")
    events = []
    for event_id, part in frame.groupby(event_column, sort=False):
        part = part.sort_values(time_column)
        if part[time_column].duplicated().any():
            raise ValueError(f"Duplicate timestamps within event {event_id}; resolve their meaning.")
        signal = part[features].apply(pd.to_numeric, errors="raise").astype(float)
        signal = signal.replace([np.inf, -np.inf], np.nan)
        signal.index = pd.DatetimeIndex(part[time_column])
        labels = None if target is None else pd.Series(
            part[target].to_numpy(), index=signal.index, name=target
        )
        group_id = event_id
        if group_column:
            if part[group_column].isna().any() or part[group_column].nunique() != 1:
                raise ValueError("Each event must have exactly one nonmissing group ID.")
            group_id = part[group_column].iloc[0]
        event_class = None if labels is None or labels.dropna().empty else labels.dropna().mode().iloc[0]
        events.append(DatasetOutputs(
            signal=signal, label=labels,
            metadata={"event_id": event_id, "group_id": group_id,
                      "event_class": event_class, "event_type": "interview",
                      "start_time": signal.index.min(), "end_time": signal.index.max()},
        ))
    return FrameDataset(events)

event_dataset = None
if df is not None and WORKFLOW == "events":
    event_dataset = events_from_frame(
        df, FEATURE_COLUMNS, TARGET_COLUMN, EVENT_COLUMN, TIMESTAMP_COLUMN, GROUP_COLUMN
    )
    print("Events:", len(event_dataset))
    display(event_dataset[0].signal.head())

### Split before fitting transforms or creating windows

Choose the strategy for the prediction task:
- **group:** hold out complete wells/entities, including all their events;
- **time:** reserve later events, with a strict non-overlap check across boundaries;
- **random:** split independent events; stratification uses the event's majority label.

Group and chronological splits are not automatically class-balanced. Check that training contains every label evaluated later. A single long series requires a deliberate time cut and possibly a gap before windowing; adapt in the blank cell rather than treating overlapping windows as independent samples.

In [ ]:
def split_indices(frame, strategy, fraction, seed, group_column=None,
                  time_column=None, stratify=None):
    if strategy is None:
        raise ValueError("Set SPLIT_STRATEGY explicitly.")
    if len(frame) < 3:
        raise ValueError("Too few observations/groups for this train/validation/test template.")
    positions = np.arange(len(frame))
    if strategy == "group":
        if not group_column or frame[group_column].isna().any():
            raise ValueError("A nonmissing GROUP_COLUMN is required.")
        splitter = GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed)
        return next(splitter.split(frame, groups=frame[group_column]))
    if strategy == "time":
        if not time_column:
            raise ValueError("A time column is required for chronological splitting.")
        order = frame[time_column].sort_values(kind="stable").index.to_numpy()
        cut = int(len(order) * (1 - fraction))
        if cut == 0 or cut == len(order):
            raise ValueError("Empty chronological partition.")
        return order[:cut], order[cut:]
    if strategy == "random":
        return train_test_split(
            positions, test_size=fraction, random_state=seed, stratify=stratify
        )
    raise ValueError("Choose group, time, or random.")

train_events = val_events = test_events = None
if event_dataset is not None and (RUN_TRAINING or RUN_TORCH):
    if TASK != "classification":
        raise ValueError("Use WORKFLOW='tabular' for this notebook's regression example.")
    if TARGET_COLUMN is None or any(
        e.label is None or e.label.isna().any() for e in event_dataset
    ):
        raise ValueError("Supervised event training needs observed labels for every sample.")
    meta = pd.DataFrame([e.metadata for e in event_dataset])
    development, test_idx = split_indices(
        meta, SPLIT_STRATEGY, TEST_FRACTION, SEED,
        "group_id", "start_time", meta["event_class"] if SPLIT_STRATEGY == "random" else None
    )
    dev_meta = meta.iloc[development].reset_index(drop=True)
    train_rel, val_rel = split_indices(
        dev_meta, SPLIT_STRATEGY, VALIDATION_FRACTION, SEED,
        "group_id", "start_time",
        dev_meta["event_class"] if SPLIT_STRATEGY == "random" else None,
    )
    train_idx, val_idx = np.asarray(development)[train_rel], np.asarray(development)[val_rel]
    train_events = SubsetDataset(event_dataset, list(train_idx))
    val_events = SubsetDataset(event_dataset, list(val_idx))
    test_events = SubsetDataset(event_dataset, list(test_idx))
    if SPLIT_STRATEGY == "group":
        group_sets = [
            {e.metadata["group_id"] for e in part}
            for part in [train_events, val_events, test_events]
        ]
        assert all(group_sets[a].isdisjoint(group_sets[b]) for a, b in [(0,1),(0,2),(1,2)])
    if SPLIT_STRATEGY == "time":
        for earlier, later in [(train_events, val_events), (val_events, test_events)]:
            if max(e.metadata["end_time"] for e in earlier) >= min(
                e.metadata["start_time"] for e in later
            ):
                raise ValueError("Time partitions overlap; choose safe boundaries/gaps.")
    train_labels = set(pd.concat([e.label for e in train_events]).unique())
    if len(train_labels) < 2:
        raise ValueError("Classification training needs at least two classes.")
    for name, partition in [("validation", val_events), ("test", test_events)]:
        unseen = set(pd.concat([e.label for e in partition]).unique()) - train_labels
        if unseen:
            raise ValueError(f"{name} contains unseen classes: {unseen}")
    print("Train / validation / test events:", len(train_events), len(val_events), len(test_events))

## 6. Toolkit preprocessing and time-series features

This example uses mean imputation fitted on training data and a learned label mapping. Signal cleaning and norm normalization are optional because they can remove useful sensor levels or invalidate small datasets.

Windowing precedes statistical features. Start padding is disabled here. Optional exponentially weighted and wavelet features mirror the reference notebooks.

FillLabelsConfig and RenameColumnsConfig are imported for deliberate use. Do not automatically infer ground truth or rename columns without documenting the mapping. Noncausal interpolation or normalization of an entire event may be unsuitable for online predictions.

In [ ]:
processor = train_processed = val_processed = test_processed = None
label_map = {}
feature_names = []
if train_events is not None:
    preprocessing_steps = []
    if USE_SIGNAL_CLEANING:
        preprocessing_steps.append(CleanSignalsConfig(missing_column_threshold=0.65))
    preprocessing_steps.append(ImputeMissingConfig(strategy="mean"))
    if USE_NORMALIZATION:
        preprocessing_steps.append(NormalizeConfig(norm="l2"))
    preprocessing_steps.append(RemapClassConfig())

    feature_steps = [StatisticalConfig(features=["mean", "std", "min", "max"])]
    if USE_EXPONENTIAL_FEATURES:
        feature_steps.append(EWStatisticalConfig(window_size=WINDOW_SIZE, decay=0.95))
    if USE_WAVELETS:
        if WINDOW_SIZE < 4 or WINDOW_SIZE & (WINDOW_SIZE - 1):
            raise ValueError("This wavelet recipe uses a power-of-two window >= 4.")
        feature_steps.append(WaveletConfig(wavelet="haar", level=int(np.log2(WINDOW_SIZE))))
    processor = TransformConfig(
        pre_processing=SequentialPreprocessingAdapterConfig(steps=preprocessing_steps),
        feature_extraction=SequentialFeatureAdapterConfig(steps=[
            WindowingConfig(
                window_size=WINDOW_SIZE, overlap=WINDOW_OVERLAP,
                label_strategy="last", pad_start=False, pad_last_window=False,
            ),
            ConcatFeatureAdapterConfig(steps=feature_steps),
        ]),
    ).build()
    processor.fit(train_events)
    train_processed = processor.transform(train_events)
    val_processed = processor.transform(val_events)
    test_processed = processor.transform(test_events)
    label_map = processor.pre_processing_step.steps[-1].class_map
    feature_names = list(train_processed[0].signal.columns)
    for name, partition in [
        ("train", train_processed), ("validation", val_processed), ("test", test_processed)
    ]:
        for event in partition:
            if event.signal.empty or not np.isfinite(event.signal.to_numpy(dtype=float)).all():
                raise ValueError(f"{name}: empty/nonfinite features; inspect lengths and missing sensors.")
            if list(event.signal.columns) != feature_names or len(event.signal) != len(event.label):
                raise ValueError(f"{name}: inconsistent features or misaligned labels.")
    print("Encoded label mapping:", label_map)
    display(train_processed[0].signal.head())

### Random Forest baseline through ThreeWToolkit

This follows the scikit-learn demo's configuration → trainer → train → predict pattern. Use validation for model decisions and reserve the test set for final assessment. Scores below are per window; consider event-level or well-level metrics for the business task.

In [ ]:
rf_trainer = rf_training = rf_predictions = rf_assessment = None
if RUN_TRAINING and train_processed is not None:
    rf_config = SklearnModelsConfig(
        model_type=RandomForestClassifier,
        model_params={"n_estimators": 100, "max_depth": 10,
                      "random_state": SEED, "class_weight": "balanced"},
    )
    rf_trainer = SklearnTrainerConfig(config_model=rf_config, seed=SEED, n_jobs=1).build()
    rf_training = rf_trainer.train(train_dataset=train_processed, val_dataset=val_processed)
    validation_predictions = rf_trainer.predict(val_processed)
    print("Validation balanced accuracy:", balanced_accuracy_score(
        validation_predictions.y_true, validation_predictions.y_pred
    ))
    rf_predictions = rf_trainer.predict(test_processed)
    assessor = ModelAssessmentConfig(
        metrics=["accuracy", "balanced_accuracy", "precision", "recall", "f1"],
        task_type=TaskTypeEnum.CLASSIFICATION, output_dir=OUTPUT_DIR,
        export_results=False, generate_report=False,
    ).build()
    rf_assessment = assessor.evaluate(training_results=rf_training, predictions=rf_predictions)
    display(pd.Series(rf_assessment.metrics, name="test_metric"))
    print(classification_report(rf_predictions.y_true, rf_predictions.y_pred, zero_division=0))

In [ ]:
if rf_assessment is not None:
    assessment_plotter = AssessmentVisualizationConfig().build()
    fig = assessment_plotter.plot_confusion_matrix(
        rf_assessment.true_values, rf_assessment.predictions, normalize=False
    )
    plt.show()
    plt.close(fig)
    fig = assessment_plotter.feature_visualization(
        rf_trainer.model.model.feature_importances_, feature_names, top_n=15
    )
    plt.show()
    plt.close(fig)

### Optional PyTorch MLP

Enable RUN_TORCH after the baseline works. This is a feed-forward classifier on window features, not an LSTM or forecast model. Its input size is inferred; output_size comes from the training-label map. The reference workshop's LSTM implementation is outside this copied package.

In [ ]:
mlp_trainer = mlp_training = mlp_predictions = mlp_assessment = None
if RUN_TORCH and train_processed is not None:
    mlp_config = MLPConfig(
        hidden_sizes=(32, 16), output_size=len(label_map),
        activation_function=nn.ReLU(),
    )
    mlp_trainer = TorchTrainerConfig(
        config_model=mlp_config, seed=SEED, use_class_weights=True,
        class_weight_strategy="balanced", learning_rate=0.001,
        batch_size=32, epochs=MLP_EPOCHS, device="cpu",
    ).build()
    mlp_training = mlp_trainer.train(train_processed, val_processed)
    mlp_predictions = mlp_trainer.predict(test_processed)
    mlp_assessment = ModelAssessmentConfig(
        metrics=["accuracy", "balanced_accuracy", "f1"],
        export_results=False, generate_report=False, output_dir=OUTPUT_DIR,
    ).build().evaluate(training_results=mlp_training, predictions=mlp_predictions)
    display(pd.Series(mlp_assessment.metrics, name="mlp_test_metric"))
    plt.plot(mlp_training.history.train_loss, label="train")
    if mlp_training.history.val_loss:
        plt.plot(mlp_training.history.val_loss, label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()
    plt.close()

**Your feature choices, model comparison, tuning, and error analysis:**

## 7. Time-series diagnostics and unsupervised analysis

FFT needs evenly sampled data and a sampling rate in Hz. Seasonal decomposition needs enough observations for at least two periods. Missing values and irregular timestamps need an explicit treatment first. These plots describe the observed series; they do not train a forecasting model.

In [ ]:
if RUN_SIGNAL_DIAGNOSTICS and example_df is not None:
    signal_column = CLUSTER_SENSOR or FEATURE_COLUMNS[0]
    series = pd.to_numeric(example_df[signal_column], errors="raise")
    if series.isna().any() or not np.isfinite(series.to_numpy()).all():
        raise ValueError("Choose an explicit missing-data treatment before signal diagnostics.")
    if isinstance(series.index, pd.DatetimeIndex):
        intervals = series.index.to_series().diff().dropna()
        if intervals.empty or intervals.nunique() != 1 or intervals.iloc[0].total_seconds() <= 0:
            raise ValueError("Signal diagnostics require regular, increasing timestamps.")
        observed_hz = 1.0 / intervals.iloc[0].total_seconds()
        if SAMPLE_RATE_HZ is not None and not np.isclose(SAMPLE_RATE_HZ, observed_hz):
            raise ValueError("SAMPLE_RATE_HZ disagrees with the timestamp interval.")
    if SAMPLE_RATE_HZ is None or SAMPLE_RATE_HZ <= 0:
        raise ValueError("Set a positive SAMPLE_RATE_HZ.")
    fig, ax = DataVisualization.plot_fft(series, sample_rate=SAMPLE_RATE_HZ)
    plt.show()
    plt.close(fig)
    fig, ax = DataVisualization.plot_wavelet_spectrogram(series)
    plt.show()
    plt.close(fig)
    if SEASONAL_PERIOD is not None:
        if SEASONAL_PERIOD < 2 or len(series) < 2 * SEASONAL_PERIOD:
            raise ValueError("Seasonal decomposition needs at least two periods of length >= 2.")
        fig, ax = DataVisualization.seasonal_decompose(
            series, model="additive", period=SEASONAL_PERIOD
        )
        plt.show()
        plt.close(fig)

### Toolkit clustering: quality → resampling → scaling → DTW → hierarchy

This branch can use unlabeled events. It compares one sensor across a bounded number of events. Per-series scaling emphasizes shape over absolute magnitude. Retain filtered event IDs when interpreting results. Cluster numbers are not fault labels.

In [ ]:
cluster_labels = None
cluster_event_ids = []
if RUN_CLUSTERING and event_dataset is not None:
    sensor = CLUSTER_SENSOR or FEATURE_COLUMNS[0]
    chosen_events = [event_dataset[i] for i in range(min(len(event_dataset), CLUSTER_EVENT_LIMIT))]
    raw_series = [e.signal[sensor].to_numpy(dtype=float) for e in chosen_events]
    quality_filter = InstanceQualityFilter(InstanceQualityConfig(
        max_nan_ratio=0.1, max_frozen_ratio=0.5
    ))
    clean_series = quality_filter.fit_transform(raw_series)
    cluster_event_ids = [chosen_events[i].metadata["event_id"] for i in quality_filter.kept_indices_]
    if len(clean_series) < 3:
        raise ValueError("Need at least three usable events for this clustering example.")
    sampled = TimeSeriesResampler(ResamplingConfig(step_size=4, step_method="mean")).fit_transform(clean_series)
    scaled = TimeSeriesScaler(TimeSeriesScalingConfig()).fit_transform(sampled)
    distance_matrix = DistanceComputer(DistanceMatrixConfig(
        metric=DistanceMetricEnum.DTW, n_jobs=1
    )).fit_transform(scaled)
    hierarchy = HierarchicalClusterer(HierarchicalClusteringConfig(
        linkage_method=LinkageMethodEnum.AVERAGE, default_threshold=0.5
    )).fit(distance_matrix)
    cluster_labels = hierarchy.get_clusters_at_threshold(0.5)
    display(pd.DataFrame({"event_id": cluster_event_ids, "cluster": cluster_labels}))
    if 1 < len(np.unique(cluster_labels)) < len(cluster_labels):
        print("Silhouette (DTW distances):", silhouette_score(
            distance_matrix, cluster_labels, metric="precomputed"
        ))
    fig, ax = DendrogramPlot(hierarchy.linkage_matrix_, threshold=0.5).plot()
    plt.show()
    plt.close(fig)

### Optional sklearn anomaly baseline

IsolationForest below operates on toolkit window features. It is a separate sklearn estimator, not a ThreeWToolkit model wrapper. Set RUN_TRAINING and RUN_ANOMALY_DETECTION together. For novelty detection, deliberately restrict training to known-normal events and calibrate thresholds on validation data; the generic example does not make that assumption.

In [ ]:
anomaly_scores = None
if RUN_ANOMALY_DETECTION and train_processed is not None:
    training_matrix = np.concatenate([e.signal.to_numpy() for e in train_processed])
    test_matrix = np.concatenate([e.signal.to_numpy() for e in test_processed])
    anomaly_model = IsolationForest(n_estimators=100, random_state=SEED, n_jobs=1)
    anomaly_model.fit(training_matrix)
    anomaly_scores = -anomaly_model.score_samples(test_matrix)
    display(pd.Series(anomaly_scores, name="larger_means_more_unusual").describe())

**Your clustering interpretation, anomaly thresholds, or custom sequence model:**

## 8. Alternative: ordinary tabular classification or regression

Select WORKFLOW="tabular" and RUN_TRAINING=True for rows that do not need event windowing. This path uses sklearn's ColumnTransformer for numeric and categorical preprocessing. It supports TASK="classification" or "regression".

Features remain explicit. Grouped/time splits use the same split helper. Numeric median imputation, scaling, and one-hot encoding are fitted on training rows only. This is a starting baseline; choose domain-appropriate transformations and metrics.

In [ ]:
tabular_pipeline = None
tabular_metrics = None
if df is not None and WORKFLOW == "tabular" and RUN_TRAINING:
    if not TARGET_COLUMN or not FEATURE_COLUMNS:
        raise ValueError("Set TARGET_COLUMN and FEATURE_COLUMNS.")
    excluded = {x for x in [TARGET_COLUMN, EVENT_COLUMN, GROUP_COLUMN, TIMESTAMP_COLUMN] if x}
    if excluded.intersection(FEATURE_COLUMNS):
        raise ValueError("Exclude target, IDs, and timestamps from feature columns.")
    if df[TARGET_COLUMN].isna().any():
        raise ValueError("Resolve missing labels before supervised learning.")
    table = df.reset_index(drop=True)
    stratify = table[TARGET_COLUMN] if TASK == "classification" and SPLIT_STRATEGY == "random" else None
    dev_idx, heldout_idx = split_indices(
        table, SPLIT_STRATEGY, TEST_FRACTION, SEED, GROUP_COLUMN, TIMESTAMP_COLUMN, stratify
    )
    development = table.iloc[dev_idx].reset_index(drop=True)
    dev_stratify = development[TARGET_COLUMN] if TASK == "classification" and SPLIT_STRATEGY == "random" else None
    train_idx, validation_idx = split_indices(
        development, SPLIT_STRATEGY, VALIDATION_FRACTION, SEED,
        GROUP_COLUMN, TIMESTAMP_COLUMN, dev_stratify,
    )
    training = development.iloc[train_idx]
    validation = development.iloc[validation_idx]
    heldout = table.iloc[heldout_idx]
    if SPLIT_STRATEGY == "time":
        if training[TIMESTAMP_COLUMN].max() >= validation[TIMESTAMP_COLUMN].min() or (
            validation[TIMESTAMP_COLUMN].max() >= heldout[TIMESTAMP_COLUMN].min()
        ):
            raise ValueError("Time boundaries share timestamps; choose a safe temporal cut.")
    numeric = [c for c in FEATURE_COLUMNS if pd.api.types.is_numeric_dtype(training[c])]
    categorical = [c for c in FEATURE_COLUMNS if c not in numeric]
    transforms = []
    if numeric:
        transforms.append(("numeric", SklearnPipeline([
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]), numeric))
    if categorical:
        transforms.append(("categorical", SklearnPipeline([
            ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical))
    if TASK == "classification":
        estimator = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=1)
        baseline = DummyClassifier(strategy="most_frequent")
    elif TASK == "regression":
        estimator = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=1)
        baseline = DummyRegressor(strategy="mean")
    else:
        raise ValueError("TASK must be classification or regression.")
    tabular_pipeline = SklearnPipeline([
        ("prepare", ColumnTransformer(transforms)), ("model", estimator)
    ])
    tabular_pipeline.fit(training[FEATURE_COLUMNS], training[TARGET_COLUMN])
    baseline.fit(np.zeros((len(training), 1)), training[TARGET_COLUMN])
    validation_output = tabular_pipeline.predict(validation[FEATURE_COLUMNS])
    predicted = tabular_pipeline.predict(heldout[FEATURE_COLUMNS])
    baseline_output = baseline.predict(np.zeros((len(heldout), 1)))
    if TASK == "classification":
        tabular_metrics = {
            "validation_balanced_accuracy": balanced_accuracy_score(validation[TARGET_COLUMN], validation_output),
            "test_balanced_accuracy": balanced_accuracy_score(heldout[TARGET_COLUMN], predicted),
            "baseline_test_balanced_accuracy": balanced_accuracy_score(heldout[TARGET_COLUMN], baseline_output),
        }
        print(classification_report(heldout[TARGET_COLUMN], predicted, zero_division=0))
    else:
        tabular_metrics = {
            "validation_mae": mean_absolute_error(validation[TARGET_COLUMN], validation_output),
            "test_mae": mean_absolute_error(heldout[TARGET_COLUMN], predicted),
            "test_rmse": float(np.sqrt(mean_squared_error(heldout[TARGET_COLUMN], predicted))),
            "test_r2": r2_score(heldout[TARGET_COLUMN], predicted),
            "baseline_test_mae": mean_absolute_error(heldout[TARGET_COLUMN], baseline_output),
        }
    display(pd.Series(tabular_metrics))

## 9. Persistence and optional reporting

Keep preprocessing, feature order, label mapping, and model together. ModelRecorder saves the toolkit model wrapper; the extra bundle preserves fitted preprocessing. Serialization should only be loaded from trusted sources.

Exports are opt-in and written under main/output/interview. A timestamped run folder avoids replacing earlier results. The sample's FrameDataset class remains notebook-local; the fitted transformer and model do not require serializing the dataset.

In [ ]:
artifact_dir = None
if SAVE_ARTIFACTS and (rf_trainer is not None or tabular_pipeline is not None or mlp_trainer is not None):
    from datetime import datetime, timezone
    artifact_dir = OUTPUT_DIR / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%f")
    artifact_dir.mkdir(parents=True, exist_ok=False)
    if rf_trainer is not None:
        ModelRecorder.save_model(rf_trainer.model, artifact_dir / "random_forest.pkl")
        joblib.dump({
            "processor": processor, "feature_names": feature_names,
            "label_map": label_map, "raw_feature_columns": FEATURE_COLUMNS,
        }, artifact_dir / "preprocessing.joblib")
    if mlp_trainer is not None:
        ModelRecorder.save_model(mlp_trainer.model, artifact_dir / "mlp.pt")
        if rf_trainer is None:
            joblib.dump({
                "processor": processor, "feature_names": feature_names,
                "label_map": label_map, "raw_feature_columns": FEATURE_COLUMNS,
            }, artifact_dir / "preprocessing.joblib")
    if tabular_pipeline is not None:
        joblib.dump(tabular_pipeline, artifact_dir / "tabular_pipeline.joblib")
    (artifact_dir / "run.json").write_text(json.dumps({
        "data_source": DATA_SOURCE, "workflow": WORKFLOW, "task": TASK,
        "seed": SEED, "features": FEATURE_COLUMNS, "target": TARGET_COLUMN,
        "split_strategy": SPLIT_STRATEGY, "toolkit_version": ThreeWToolkit.__version__,
    }, indent=2))
    print("Saved:", artifact_dir)

### Reports and complete pipeline configuration

ReportGeneration and PipelineConfig are imported for reuse. The explicit stages above are easier to adapt before the interview schema is known.

- ReportGeneration takes model, train_len, test_len, predictions, calculated_metrics, plot_config, title, and optional author/output settings.
- PipelineConfig requires train_dataset_config and trainer_config; optionally supply validation/test dataset configs, transform_config, assessment_config, and task.
- The toolkit has HTML and LaTeX assets. Start with HTML; LaTeX/PDF requires additional system tools.
- The configuration-driven pipeline expects a BaseDatasetConfig. Our small notebook adapter is an in-memory BaseDataset; add a matching config if migrating it into PipelineConfig.

**Optional HTML report from the Random Forest results:**

In [ ]:
if GENERATE_HTML_REPORT and rf_assessment is not None:
    from datetime import datetime, timezone
    report_title = "interview_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%f")
    report_builder = ReportGeneration(
        model=rf_trainer.model,
        train_len=len(train_processed), test_len=len(test_processed),
        predictions=pd.Series(rf_predictions.y_pred),
        calculated_metrics=rf_assessment.metrics, plot_config={},
        title=report_title, author="Interview preparation",
        reports_dir=OUTPUT_DIR / "reports",
    )
    html_report = report_builder.generate_summary_report(format="html")
    report_builder.save_report(html_report, filename="summary", format="html")

## Reference notebook map

These references stay in the original 3W project; they are not dependencies of this starter.

| Topic | Reference |
|---|---|
| Main supervised workflow | [sklearn_models_examples](../Stage_1_learn_data/3W/toolkit/demos/sklearn_models_examples.ipynb) |
| Cleaning and preprocessing | [data_cleaning_examples](../Stage_1_learn_data/3W/toolkit/demos/data_cleaning_examples.ipynb), [preprocessing_examples](../Stage_1_learn_data/3W/toolkit/demos/preprocessing_examples.ipynb) |
| Features | [feature_extraction_examples](../Stage_1_learn_data/3W/toolkit/demos/feature_extraction_examples.ipynb) |
| Visualization | [data_visualization](../Stage_1_learn_data/3W/toolkit/demos/data_visualization.ipynb), [three_w_chart_example](../Stage_1_learn_data/3W/toolkit/demos/three_w_chart_example.ipynb) |
| Neural networks | [mlp_timeseries_example](../Stage_1_learn_data/3W/toolkit/demos/mlp_timeseries_example.ipynb) |
| Clustering | [clustering_pipeline_examples](../Stage_1_learn_data/3W/toolkit/demos/clustering_pipeline_examples.ipynb), [real_vs_simulated](../Stage_1_learn_data/3W/toolkit/demos/clustering_real_vs_simulated_examples.ipynb) |
| Metrics and evaluation | [using_metrics](../Stage_1_learn_data/3W/toolkit/demos/using_metrics.ipynb), [assessment_examples](../Stage_1_learn_data/3W/toolkit/demos/assessment_examples.ipynb) |
| Persistence and reports | [load_save_model_examples](../Stage_1_learn_data/3W/toolkit/demos/load_save_model_examples.ipynb), [report_generation_examples](../Stage_1_learn_data/3W/toolkit/demos/report_generation_examples.ipynb) |
| Broader ML workshop | [workshop folder](../Stage_1_learn_data/3W/resources/introduction_to_ml_applied_to_mts/) — separate scripts for SVM/neural comparisons, LSTM autoencoding, t-SNE/UMAP, and clustering |
| Full toolkit tour | [overview](../Stage_1_learn_data/3W/toolkit/demos/overview.ipynb) |

Current imports were checked against the copied package. Older v2 examples contain different APIs; do not paste their imports without checking.

## Interview findings and next actions

- What did the data represent?
- Which reusable components did you select, and why?
- What did you implement or adapt yourself?
- What prevented leakage between training and evaluation?
- Which baseline and metric answered the business question?
- Which errors mattered most, and what would you improve?
- What remains untested or depends on missing data?

Keep synthetic checks separate from real results. Record final decisions here.